# sim_dual_arm — Google Colab 数据转换与 ACT 训练流程（rclone 版）

本 Notebook 通过 **VS Code Colab 插件**（或网页端 Colab）连接云端运行时执行 **大内存数据转换** 与 **ACT 模型训练**，所有核心逻辑位于 [`docs/colab/scripts/`](scripts/)，Notebook 仅保留参数与入口调用。

## 前置步骤（在本机执行，二选一）

- **方式 A：云端大内存转换 + 训练（推荐）**
  1. 本机采集达标生成 `output/<robot>/<backend>/<task>/datasets/raw/formal-select.json`。
  2. 在配套同步 Notebook [sync_drive_rclone.ipynb](sync_drive_rclone.ipynb)（选择本机 Python 内核）修改 `ROBOT`、`BACKEND`、`TASK` 并运行 **第 3 节**，将仅含合格 `export/`（不含 MCAP）的 `raw_exports.tar` 上传至 Google Drive。
- **方式 B：本机已转换好数据集，仅上云训练**
  - 在 [sync_drive_rclone.ipynb](sync_drive_rclone.ipynb) 运行 **第 4 节**上传 `lerobot.tar`。

## 使用步骤

1. 在 VS Code 右上角点击 **Select Kernel** → **Colab** → 推荐选择 **高 RAM（High-RAM）GPU 运行时**（L4 / A100 / 高 RAM T4）；若只想低成本单独执行数据转换，也可用 **高 RAM CPU 运行时**运行至第 5 节自动转换并回传 `lerobot.tar`，随后切换 GPU 运行时训练。
2. 在下方 **第 1 节** 按需修改 `ROBOT`、`BACKEND`、`TASK`、`CONVERT_MODE`（默认 `"auto"` 自动对比云端时间戳）、`BASE_STEPS`、`AUTO_SCALE_STEPS` 与视觉增强开关。
3. 按顺序运行 Cell：
   - **第 2 节**：克隆或更新仓库并加载脚本模块；
   - **第 3 节**：检查 GPU/CPU/内存并配置 `rclone`（免 `drive.mount`）；
   - **第 4 节**：安装学习端依赖并复用意向缓存；
   - **第 5 节**：按 `CONVERT_MODE` 自动决策——若需要转换则利用 `/dev/shm` 内存盘加速转换并自动回传 `lerobot.tar` 与转换日志至 Google Drive，否则直接拉取最新 `lerobot.tar`；
   - **第 6 节**：探测当前 GPU 最优训练参数；
   - **第 7 节**：按探测 `batch_size` 自动缩放到 1000 的倍数步数并每 1000 步刷新训练/验证损失曲线图；
   - **第 8 节**：将模型、训练日志与损失图回传 Google Drive。
4. 训练完成后，在本机打开 [sync_drive_rclone.ipynb](sync_drive_rclone.ipynb) 运行 **第 6 节**即可一键拉回模型与日志。


In [ ]:
# 1. 参数配置（按需修改后运行）
REPO_URL = "https://github.com/ljk4/sim_dual_arm.git"
REPO_BRANCH = "main"
WORK_DIR = "/content/sim_dual_arm"
RCLONE_REMOTE = "gdrive:sim_dual_arm"
USE_COLAB_SECRETS = False  # 仅网页端已配置 Colab Secrets 且需自动读取时改为 True

# 按需切换任务：如 marvin / mujoco / stack 或 marvin / mujoco / handover
ROBOT = "marvin"
BACKEND = "mujoco"
TASK = "handover"
TRAIN_MODE = "full"       # 可选: smoke / overfit / full
RAW_ARCHIVE = "raw_exports.tar"
DATASET_ARCHIVE = "lerobot.tar"
CONVERT_MODE = "auto"         # "auto": 云端有更新原始数据时自动转换; "convert": 强制转换; "fetch": 仅拉取 lerobot.tar
USE_SHM_FOR_CONVERT = True    # 内存充足时优先将原始导出解压到 /dev/shm 内存盘加速转换
CLEANUP_RAW_AFTER_CONVERT = True  # 转换完成后立即清理原始导出以释放内存盘和磁盘空间
SYNC_CONVERTED_TO_DRIVE = True    # 转换完成后自动将 lerobot.tar 与转换日志上传回 Google Drive
USE_DRIVE_CACHE = True
TORCH_INDEX_URL = ""      # 留空自动按显卡驱动选择 cu128 / cu126

# 步数与 batch 自适应策略（full 模式：以 batch=32 为基准，步数向上取整为 1000 的倍数）
BASE_STEPS = 20000            # 参考 batch_size=32 时的基准总步数
REFERENCE_BATCH_SIZE = 32     # 基准 batch_size
AUTO_SCALE_STEPS = True       # True 时按 batch 比例反比缩放步数并向上取整到 STEP_MULTIPLE 倍数
STEP_MULTIPLE = 1000          # 步数取整粒度
PLOT_EVERY_STEPS = 1000       # 每隔多少步绘制并刷新训练/验证损失曲线
MAX_STEPS = None              # 若填写整数，则直接以该步数覆盖自适应计算

# 视觉增强开关覆盖（None 表示沿用配方默认值：enabled=True, camera_dropout_prob=0.10）
VISUAL_AUGMENT = None         # 可设为 True / False / None
CAMERA_DROPOUT_PROB = None    # 可设为 0.0 ~ 1.0 浮点数或 None

# 服务器资源最大化探测与覆盖选项
AUTO_PROBE = True             # True 时在第 6 节实测最优 batch_size / num_workers / prefetch_factor
MAX_BATCH_SIZE = 256          # 探测 batch_size 上限（受限于 TrainingConfig <= 256）
VRAM_TARGET_RATIO = 0.85      # 允许占用的最大显存比例
RAM_TARGET_RATIO = 0.70       # 允许占用的最大可用内存 / /dev/shm 比例
BATCH_SIZE = None             # 若显式填写整数，将覆盖探测/配方值
NUM_WORKERS = None            # 若显式填写整数，将覆盖探测/配方值
PREFETCH_FACTOR = None        # 若显式填写整数，将覆盖探测/配方值
SCALE_LR_SQRT = False         # True 时按 sqrt(batch_size / base_batch_size) 缩放 lr / lr_backbone

print(
    f"配置已就绪: {ROBOT}/{BACKEND}/{TASK} | mode={TRAIN_MODE} | convert_mode={CONVERT_MODE} | "
    f"base_steps={BASE_STEPS}@{REFERENCE_BATCH_SIZE} | auto_scale={AUTO_SCALE_STEPS} | "
    f"plot_every={PLOT_EVERY_STEPS} | max_steps={MAX_STEPS} | remote={RCLONE_REMOTE} | "
    f"visual_aug={VISUAL_AUGMENT} | cam_drop={CAMERA_DROPOUT_PROB} | auto_probe={AUTO_PROBE}"
)

In [ ]:
# 2. 克隆或更新公开仓库并加载 Colab 工作流模块
import subprocess
import sys
from pathlib import Path

work_path = Path(WORK_DIR)
if (work_path / ".git").is_dir():
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, WORK_DIR],
        check=True,
    )

if str(work_path) not in sys.path:
    sys.path.insert(0, str(work_path))

from docs.colab.scripts.colab_workflow import (
    prepare_dataset,
    probe_hardware,
    run_training,
    setup_learning_env,
    sync_results_to_drive,
)
from docs.colab.scripts.rclone_auth import setup_colab_rclone

commit = subprocess.run(
    ["git", "-C", WORK_DIR, "log", "-1", "--oneline"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"仓库就绪: {commit}")

In [ ]:
# 3. 检查 GPU 并配置 rclone（免 drive.mount）
setup_colab_rclone(
    rclone_remote=RCLONE_REMOTE,
    robot=ROBOT,
    backend=BACKEND,
    task=TASK,
    use_colab_secrets=USE_COLAB_SECRETS,
)

In [ ]:
# 4. 安装学习端环境与第三方源码/权重（通过 rclone 复用 _cache）
setup_learning_env(
    work_dir=WORK_DIR,
    rclone_remote=RCLONE_REMOTE,
    use_drive_cache=USE_DRIVE_CACHE,
    torch_index_url=TORCH_INDEX_URL,
)

In [ ]:
# 5. 准备数据集（自动对比云端时间戳：执行大内存转换并回传 lerobot.tar，或直接拉取已有数据集）
prepare_dataset(
    work_dir=WORK_DIR,
    rclone_remote=RCLONE_REMOTE,
    robot=ROBOT,
    backend=BACKEND,
    task=TASK,
    convert_mode=CONVERT_MODE,
    raw_archive=RAW_ARCHIVE,
    dataset_archive=DATASET_ARCHIVE,
    use_shm_for_convert=USE_SHM_FOR_CONVERT,
    cleanup_raw_after_convert=CLEANUP_RAW_AFTER_CONVERT,
    sync_converted_to_drive=SYNC_CONVERTED_TO_DRIVE,
)

In [ ]:
# 6. 探测当前服务器最优训练参数（最大化利用 GPU 显存与 CPU/RAM）
probe_hardware(
    work_dir=WORK_DIR,
    robot=ROBOT,
    backend=BACKEND,
    task=TASK,
    mode=TRAIN_MODE,
    auto_probe=AUTO_PROBE,
    max_batch_size=MAX_BATCH_SIZE,
    vram_target_ratio=VRAM_TARGET_RATIO,
    ram_target_ratio=RAM_TARGET_RATIO,
)

In [ ]:
# 7. 启动 ACT 训练（按 batch 自动缩放步数并每 1000 步输出训练/验证损失图）
run_training(
    work_dir=WORK_DIR,
    robot=ROBOT,
    backend=BACKEND,
    task=TASK,
    mode=TRAIN_MODE,
    base_steps=BASE_STEPS,
    reference_batch_size=REFERENCE_BATCH_SIZE,
    auto_scale_steps=AUTO_SCALE_STEPS,
    step_multiple=STEP_MULTIPLE,
    plot_every_steps=PLOT_EVERY_STEPS,
    max_steps=MAX_STEPS,
    auto_probe=AUTO_PROBE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    scale_lr_sqrt=SCALE_LR_SQRT,
    visual_augment=VISUAL_AUGMENT,
    camera_dropout_prob=CAMERA_DROPOUT_PROB,
)

In [ ]:
# 8. 通过 rclone 同步最新模型、训练日志与损失图回 Google Drive
sync_results_to_drive(
    work_dir=WORK_DIR,
    rclone_remote=RCLONE_REMOTE,
    robot=ROBOT,
    backend=BACKEND,
    task=TASK,
)